### 06 — Rotten Tomatoes — Avaliações — Camada Prata

Objetivo

Transformar os dados de avaliações de críticos do Rotten Tomatoes,
armazenados na camada Bronze, em dados tratados e estruturados na
camada Prata.

Transformações realizadas

- remoção de registros duplicados;
- tratamento de espaços em campos textuais;
- conversão da data da avaliação;
- padronização do indicador de crítico principal;
- padronização da classificação da avaliação;
- preservação do score original;
- criação de identificador técnico da avaliação;
- validação da integridade referencial com os filmes;
- gravação da tabela tratada em formato Delta.

Decisão sobre `review_score`

O campo `review_score` apresenta diferentes escalas e formatos,
como frações, notas numéricas e conceitos alfabéticos.

Por esse motivo, seu valor original será preservado na camada Prata.
Uma eventual normalização para escala comum deverá ser realizada
posteriormente.

In [0]:
from pyspark.sql import functions as F

In [0]:
TABELA_AVALIACOES_BRONZE = (
    "mvp3_eng_dados.bronze.rt_avaliacoes"
)

TABELA_FILMES_PRATA = (
    "mvp3_eng_dados.prata.rt_filmes"
)

TABELA_AVALIACOES_PRATA = (
    "mvp3_eng_dados.prata.rt_avaliacoes"
)

print(
    "Origem: ",
    TABELA_AVALIACOES_BRONZE
)

print(
    "Referência: ",
    TABELA_FILMES_PRATA
)

print(
    "Destino: ",
    TABELA_AVALIACOES_PRATA
)

Origem:  mvp3_eng_dados.bronze.rt_avaliacoes
Referência:  mvp3_eng_dados.prata.rt_filmes
Destino:  mvp3_eng_dados.prata.rt_avaliacoes


In [0]:
df_rt_avaliacoes_bronze = spark.table(
    TABELA_AVALIACOES_BRONZE
)

qtd_bronze = (
    df_rt_avaliacoes_bronze
    .count()
)

print(
    f"Registros Bronze: "
    f"{qtd_bronze:,}"
)

df_rt_avaliacoes_bronze.printSchema()

Registros Bronze: 1,130,017
root
 |-- rotten_tomatoes_link: string (nullable = true)
 |-- critic_name: string (nullable = true)
 |-- top_critic: string (nullable = true)
 |-- publisher_name: string (nullable = true)
 |-- review_type: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_date: string (nullable = true)
 |-- review_content: string (nullable = true)
 |-- _fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)



In [0]:
colunas_funcionais = [
    "rotten_tomatoes_link",
    "critic_name",
    "top_critic",
    "publisher_name",
    "review_type",
    "review_score",
    "review_date",
    "review_content"
]

df_rt_avaliacoes_sem_duplicados = (
    df_rt_avaliacoes_bronze
    .select(
        *colunas_funcionais
    )
    .dropDuplicates()
)

qtd_sem_duplicados = (
    df_rt_avaliacoes_sem_duplicados
    .count()
)

print(
    f"Registros Bronze: "
    f"{qtd_bronze:,}"
)

print(
    "Registros após deduplicação: "
    f"{qtd_sem_duplicados:,}"
)

print(
    "Duplicados removidos: "
    f"{qtd_bronze - qtd_sem_duplicados:,}"
)

Registros Bronze: 1,130,017
Registros após deduplicação: 1,010,546
Duplicados removidos: 119,471


In [0]:
colunas_textuais = [
    "rotten_tomatoes_link",
    "critic_name",
    "publisher_name",
    "review_type",
    "review_score",
    "review_date",
    "review_content"
]

df_rt_avaliacoes_prata = (
    df_rt_avaliacoes_sem_duplicados
)

for coluna in colunas_textuais:
    df_rt_avaliacoes_prata = (
        df_rt_avaliacoes_prata
        .withColumn(
            coluna,
            F.when(
                F.trim(
                    F.col(coluna)
                ) == "",
                None
            )
            .otherwise(
                F.trim(
                    F.col(coluna)
                )
            )
        )
    )

In [0]:
df_rt_avaliacoes_prata = (
    df_rt_avaliacoes_prata
    .withColumn(
        "eh_top_critic",
        F.when(
            F.lower(
                F.col("top_critic")
                .cast("string")
            ).isin(
                "true",
                "1",
                "yes"
            ),
            True
        )
        .when(
            F.lower(
                F.col("top_critic")
                .cast("string")
            ).isin(
                "false",
                "0",
                "no"
            ),
            False
        )
        .otherwise(
            None
        )
    )
)

In [0]:
df_rt_avaliacoes_prata = (
    df_rt_avaliacoes_prata
    .withColumn(
        "data_avaliacao",
        F.to_date(
            F.col("review_date")
        )
    )
)

In [0]:
df_rt_avaliacoes_prata = (
    df_rt_avaliacoes_prata
    .withColumn(
        "sentimento",
        F.when(
            F.lower(
                F.col("review_type")
            ) == "fresh",
            F.lit("POSITIVO")
        )
        .when(
            F.lower(
                F.col("review_type")
            ) == "rotten",
            F.lit("NEGATIVO")
        )
        .otherwise(
            F.lit("DESCONHECIDO")
        )
    )
)

In [0]:
df_rt_avaliacoes_prata = (
    df_rt_avaliacoes_prata
    .withColumn(
        "id_avaliacao",
        F.sha2(
            F.concat_ws(
                "||",

                F.coalesce(
                    F.col(
                        "rotten_tomatoes_link"
                    ),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("critic_name"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("top_critic")
                    .cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("publisher_name"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("review_type"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("review_score"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("review_date"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("review_content"),
                    F.lit("")
                )
            ),
            256
        )
    )
)

In [0]:
df_rt_avaliacoes_prata = (
    df_rt_avaliacoes_prata
    .withColumn(
        "fonte_dados",
        F.lit("Rotten Tomatoes")
    )
    .withColumn(
        "data_hora_processamento",
        F.current_timestamp()
    )
    .select(
        "id_avaliacao",
        "rotten_tomatoes_link",
        "critic_name",
        "top_critic",
        "eh_top_critic",
        "publisher_name",
        "review_type",
        "sentimento",
        "review_score",
        "review_date",
        "data_avaliacao",
        "review_content",
        "fonte_dados",
        "data_hora_processamento"
    )
)

df_rt_avaliacoes_prata.printSchema()

root
 |-- id_avaliacao: string (nullable = true)
 |-- rotten_tomatoes_link: string (nullable = true)
 |-- critic_name: string (nullable = true)
 |-- top_critic: string (nullable = true)
 |-- eh_top_critic: boolean (nullable = true)
 |-- publisher_name: string (nullable = true)
 |-- review_type: string (nullable = true)
 |-- sentimento: string (nullable = false)
 |-- review_score: string (nullable = true)
 |-- review_date: string (nullable = true)
 |-- data_avaliacao: date (nullable = true)
 |-- review_content: string (nullable = true)
 |-- fonte_dados: string (nullable = false)
 |-- data_hora_processamento: timestamp (nullable = false)



In [0]:
qtd_prata = (
    df_rt_avaliacoes_prata
    .count()
)

ids_nulos = (
    df_rt_avaliacoes_prata
    .filter(
        F.col("id_avaliacao").isNull()
    )
    .count()
)

ids_distintos = (
    df_rt_avaliacoes_prata
    .select("id_avaliacao")
    .distinct()
    .count()
)

sentimentos_desconhecidos = (
    df_rt_avaliacoes_prata
    .filter(
        F.col("sentimento")
        == "DESCONHECIDO"
    )
    .count()
)

print(
    f"Registros Prata: "
    f"{qtd_prata:,}"
)

print(
    f"IDs nulos: "
    f"{ids_nulos:,}"
)

print(
    f"IDs distintos: "
    f"{ids_distintos:,}"
)

print(
    "Sentimentos desconhecidos: "
    f"{sentimentos_desconhecidos:,}"
)

assert ids_nulos == 0, (
    "ERRO: existem IDs nulos."
)

assert ids_distintos == qtd_prata, (
    "ERRO: id_avaliacao não é único. "
    f"Registros={qtd_prata:,}; "
    f"IDs distintos={ids_distintos:,}."
)

assert sentimentos_desconhecidos == 0, (
    "ERRO: existem sentimentos "
    "não mapeados."
)

print(
    "\n✓ Validações estruturais aprovadas."
)

Registros Prata: 1,010,546
IDs nulos: 0
IDs distintos: 1,010,546
Sentimentos desconhecidos: 0

✓ Validações estruturais aprovadas.


In [0]:
datas_invalidas = (
    df_rt_avaliacoes_prata
    .filter(
        F.col("review_date").isNotNull()
        &
        F.col("data_avaliacao").isNull()
    )
    .count()
)

top_critic_invalidos = (
    df_rt_avaliacoes_prata
    .filter(
        F.col("top_critic").isNotNull()
        &
        F.col("eh_top_critic").isNull()
    )
    .count()
)

print(
    "Datas não convertidas: "
    f"{datas_invalidas:,}"
)

print(
    "Valores top_critic não convertidos: "
    f"{top_critic_invalidos:,}"
)

Datas não convertidas: 0
Valores top_critic não convertidos: 0


In [0]:
df_rt_filmes_prata = (
    spark.table(
        TABELA_FILMES_PRATA
    )
    .select(
        "rotten_tomatoes_link"
    )
    .dropDuplicates()
)

df_avaliacoes_sem_filme = (
    df_rt_avaliacoes_prata
    .select(
        "rotten_tomatoes_link"
    )
    .filter(
        F.col(
            "rotten_tomatoes_link"
        ).isNotNull()
    )
    .dropDuplicates()
    .join(
        df_rt_filmes_prata,
        on="rotten_tomatoes_link",
        how="left_anti"
    )
)

qtd_avaliacoes_sem_filme = (
    df_avaliacoes_sem_filme
    .count()
)

print(
    "Links de avaliações sem filme "
    "correspondente: "
    f"{qtd_avaliacoes_sem_filme:,}"
)

Links de avaliações sem filme correspondente: 6


In [0]:
display(
    df_rt_avaliacoes_prata
    .groupBy(
        "review_type",
        "sentimento"
    )
    .count()
    .orderBy(
        "sentimento"
    )
)

review_type,sentimento,count
Rotten,NEGATIVO,366614
Fresh,POSITIVO,643932


In [0]:
(
    df_rt_avaliacoes_prata
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TABELA_AVALIACOES_PRATA
    )
)

print(
    "Tabela gravada com sucesso:"
)

print(
    TABELA_AVALIACOES_PRATA
)

Tabela gravada com sucesso:
mvp3_eng_dados.prata.rt_avaliacoes


In [0]:
df_rt_avaliacoes_validacao = (
    spark.table(
        TABELA_AVALIACOES_PRATA
    )
)

qtd_gravada = (
    df_rt_avaliacoes_validacao
    .count()
)

ids_gravados_distintos = (
    df_rt_avaliacoes_validacao
    .select(
        "id_avaliacao"
    )
    .distinct()
    .count()
)

assert qtd_gravada == qtd_prata, (
    "ERRO: quantidade gravada "
    "difere do DataFrame Prata."
)

assert (
    ids_gravados_distintos
    == qtd_gravada
), (
    "ERRO: tabela persistida "
    "possui IDs repetidos."
)

print(
    f"Registros gravados: "
    f"{qtd_gravada:,}"
)

print(
    "IDs distintos gravados: "
    f"{ids_gravados_distintos:,}"
)

print(
    "\n✓ Persistência validada."
)

Registros gravados: 1,010,546
IDs distintos gravados: 1,010,546

✓ Persistência validada.


In [0]:
display(
    df_rt_avaliacoes_validacao
    .select(
        "id_avaliacao",
        "rotten_tomatoes_link",
        "critic_name",
        "publisher_name",
        "review_type",
        "sentimento",
        "review_score",
        "data_avaliacao",
        "review_content"
    )
    .limit(10)
)

id_avaliacao,rotten_tomatoes_link,critic_name,publisher_name,review_type,sentimento,review_score,data_avaliacao,review_content
df86ce69451f840281b0b4845f1b48650e0335b3fcde5d0df402865eebdf1a4d,m/0814255,Mark Pfeiffer,Reel Times: Reflections on Cinema,Rotten,NEGATIVO,C+,2010-02-16,Percy Jackson & the Olympians is a dutifully made kids' fantasy film that Columbus handles with the basic approach that he wielded on his faithful but vanilla Harry Potter adaptations.
ea6e7464a2878d7174ec3e968337382cebbb903e97f90e783b2c0be7d769ca9b,m/0814255,Charles Gant,Heat Magazine,Fresh,POSITIVO,3/5,2010-02-16,"Not bad, but JK Rowling can rest easy."
8fa5e34d196253cdd8b18863ea6ff977951025caf7217bbcf2f9a249e1424b94,m/0814255,Elliott Noble,Sky Cinema,Fresh,POSITIVO,3/5,2010-02-16,There's plenty of larks to be had from a fast-moving fantasy that doesn't take its mythology too seriously and employs actors who don't look like teenage rabbits in headlights.
3673aae86810a3455ded0eadcdd6e93c460385679326dd35e0f4f2ca004e11ec,m/0878835,Mike Scott,Times-Picayune,Fresh,POSITIVO,3/4,2010-06-30,"A well-written, well-acted dramatic comedy that is light on plot but generous in spirit."
2cd6eabd656c873d1b33ab5ccfcf648f3b8f50d4ddc5150ce0fa3e636a5c7a3e,m/10000_bc,Eric Melin,Scene-Stealers.com,Rotten,NEGATIVO,1/4,2008-03-11,"Making prehistoric people talk and behave just like modern people in a 10,000 B.C. setting makes actors look silly. Their costumes are silly, the settings are silly, and the whole movie is unintentionally funny."
f5f7982eac0fe2d1e45ecb67f11be3fc5133a67ac8e7d9c4b8f3befc5ac94a99,m/10000_bc,Felix Vasquez Jr.,Film Threat,Rotten,NEGATIVO,2/5,2008-03-13,"Strait has to be one of the most unsympathetic cardboard heroes of the genre, while Bell leaves little reason to care for her beyond her bitching dreadlocks..."
25a35efce08bb1586370baa2d34ae555904378be2c73cb7d9ddf0337b2c600a9,m/1000121-39_steps,Helen Brown Norden,Vanity Fair,Fresh,POSITIVO,null,2019-06-12,"The newest Gaumont British importation, The 39 Steps, should prove pretty conclusively that Alfred Hitchcock is the finest native director in England."
c17789667d1273231a40d8758c7dab2165f3efbdc3fbefbfc36fe610ec8a7e10,m/10002008-charly,Audrey Rock-Richardson,Tooele Transcript-Bulletin (Utah),Rotten,NEGATIVO,C,2002-10-07,Charly comes off as emotionally manipulative and sadly imitative of innumerable past Love Story derisions.
17e3efee48086e4ed4b17467961df97765d6e457c8be90b1a6bb8a5780d2f1bf,m/1000224-accused,Cole Smithey,ColeSmithey.com,Fresh,POSITIVO,4/5,2006-03-13,null
4c8b92ac1ad67b5ed6b894bd633c8a0a90dd5869d72274fa86076f86365cfe8b,m/1000355-adventures_of_robin_hood,James Plath,Movie Metropolis,Fresh,POSITIVO,5/5,2005-09-28,A flawless swashbuckler!


In [0]:
print("=" * 60)

print(
    "RT AVALIAÇÕES — "
    "CAMADA PRATA CONCLUÍDA"
)

print("=" * 60)

print(
    f"Bronze: "
    f"{qtd_bronze:,}"
)

print(
    f"Prata: "
    f"{qtd_gravada:,}"
)

print(
    "Duplicados removidos: "
    f"{qtd_bronze - qtd_gravada:,}"
)

print(
    "IDs distintos: "
    f"{ids_gravados_distintos:,}"
)

print(
    "Links de avaliações sem filme: "
    f"{qtd_avaliacoes_sem_filme:,}"
)

print(
    "\nTransformações realizadas:"
)

print(
    "✓ duplicidades exatas tratadas"
)

print(
    "✓ campos textuais tratados"
)

print(
    "✓ data convertida"
)

print(
    "✓ top_critic convertido"
)

print(
    "✓ sentimento padronizado"
)

print(
    "✓ review_score original preservado"
)

print(
    "✓ identificador técnico único criado"
)

print(
    "✓ integridade referencial avaliada"
)

print(
    "✓ tabela Delta gravada"
)

print(
    "✓ persistência validada"
)

RT AVALIAÇÕES — CAMADA PRATA CONCLUÍDA
Bronze: 1,130,017
Prata: 1,010,546
Duplicados removidos: 119,471
IDs distintos: 1,010,546
Links de avaliações sem filme: 6

Transformações realizadas:
✓ duplicidades exatas tratadas
✓ campos textuais tratados
✓ data convertida
✓ top_critic convertido
✓ sentimento padronizado
✓ review_score original preservado
✓ identificador técnico único criado
✓ integridade referencial avaliada
✓ tabela Delta gravada
✓ persistência validada
